# Introduction
Introduction to Databricks Features such as its notebook, file-management, git integration, volumes, table, path, and more!


---

## Data Reading

### #1: JSON Data

In [0]:
df_json = spark.read.format("json").option("inferSchema", True).option("header", True).option("multiLine", False).load("/Volumes/workspace/introduction/raw-data/drivers.json")

In [0]:
display(df_json)

### #2: CSV Data

In [0]:
df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load("/Volumes/workspace/introduction/raw-data/BigMart Sales.csv")
)

In [0]:
display(df)

---

## SCHEMA – DDL and StructType()

In [0]:
df.printSchema()

### Cara 01: DDL

In [0]:
# Kita akan mengubah dtype dari Item_Weight (double) menjadi string
my_ddl_schema = """
    Item_Identifier STRING,
    Item_Weight STRING,
    Item_Fat_Content STRING,
    Item_Visibility DOUBLE,
    Item_Type STRING,
    Item_MRP DOUBLE,
    Outlet_Identifier STRING,
    Outlet_Establishment_Year INTEGER,
    Outlet_Size STRING,
    Outlet_Location_Type STRING,
    Outlet_Type STRING,
    Item_Outlet_Sales DOUBLE
"""

In [0]:
# Read lagi file csv nya menggunakan schema (DDL) yang kita buat secara manual
df_ddl = spark.read.csv(
    "/Volumes/workspace/introduction/raw-data/BigMart Sales.csv",
    header=True,
    schema=my_ddl_schema
)

In [0]:
display(df_ddl)

### Cara 2: StructType - StructField

In [0]:
# Kita akan merubah SEMUA dtype dari df_spark menjadi STRING
from pyspark.sql.types import *

my_struct_schema = StructType([
    StructField('Item_Identifier', StringType(), True),
    StructField('Item_Weight', StringType(), True),
    StructField('Item_Fat_Content', StringType(), True),
    StructField('Item Visibility', StringType(), True),
    StructField('Item_MRP', StringType(), True),
    StructField('Outlet_Identifier', StringType(), True),
    StructField('Outlet_Establishment_Year', StringType(), True),
    StructField('Outlet_Size', StringType(), True),
    StructField('Outlet_Location_Type', StringType(), True),
    StructField('Outlet_Type', StringType(), True),
    StructField('Item_Outlet_Sales', StringType(), True)
])

In [0]:
df_struct = (
    spark.read
    .format("csv")
    .option("header", True)
    .schema(my_struct_schema)
    .load("/Volumes/workspace/introduction/raw-data/BigMart Sales.csv")
)

display(df_struct)

---

## Beginner Data Transformation with Spark

### SELECT

In [0]:
df_sel = df.select("Item_Identifier", "Item_Weight", "Item_Fat_Content")

In [0]:
from pyspark.sql.functions import col
df_sel = df.select(col("Item_Identifier"), col("Item_Weight"), col("Item_Fat_Content"))
display(df_sel)

### ALIAS

In [0]:
df_alias = df.select(
    col("Item_Identifier").alias("item_id"),
    col("Item_Weight").alias("item_kg")
)

In [0]:
display(df_alias)

### FILTERING

In [0]:
# filter() dan where() adalah method yang sama fungsinya.
# filter() memilih baris, sedangkan select() memilih kolom.
# Kondisi dapat ditulis menggunakan Column object (col("salary") > 5000) atau string SQL ("salary > 5000").
# Gunakan & untuk AND, | untuk OR, dan ~ untuk NOT, bukan and, or, atau not.
# Selalu beri tanda kurung pada setiap kondisi ketika menggunakan & atau |.
# Method yang sering dipakai bersama filter() antara lain isin(), between(), like(), startswith(), isNull(), dan isNotNull().
# Dalam praktik, filter() sering ditempatkan di awal pipeline agar jumlah data yang diproses oleh operasi berikutnya menjadi lebih sedikit dan lebih efisien.

# TUGAS:
#     1) Filter the data with fat content = Regular
#     2) Slice the data with item type = Soft Drinks and weight < 10
#     3) Fetch the data with Tier in (Tier1 or Tier 2) and Outlet Size is Null

In [0]:
# TUGAS 1
df_task1 = df.filter(col("Item_Fat_Content") == "Regular")
display(df_task1)

In [0]:
# TUGAS 2
df_task2 = df.filter(
    (col("Item_Type") == "Soft Drinks") &
    (col("Item_Weight") < 10)
)

display(df_task2)

In [0]:
# Tugas 3
df_task3 = df.filter(
    (col("Outlet_Location_Type")).isin("Tier 1", "Tier 2") &
    (col("Outlet_Size").isNull())
)
display(df_task3)

### withColumnRenamed

In [0]:
# withColumnRenamed --> hanya mengganti nama kolom
df_new_name = df.withColumnRenamed("Item_Identifier", "Item_Id").withColumnRenamed("Item_Weight", "Item_Kg")
display(df_new_name)

### withColumn
membuat kolom baru atau mengganti isi kolom yang sudah ada.

#### Scenario - 1

In [0]:
# Scenario 1: Create a new column within the dataframe
from pyspark.sql.functions import lit

# lit() → membuat nilai tetap yang sama (konstan) untuk setiap baris.
df_new_col = df.withColumn("Wuwa", lit("Suisui menang 50/50 Aaamiinn"))

# Membuat column baru dari hasil operasi suatu column
df_new_col = df_new_col.withColumn("Item_Weight_Gram", col("Item_Weight") * 1000)

display(df_new_col)

#### Scenario - 2

In [0]:
# Scenario 2: Change the content of the existing column
from pyspark.sql.functions import regexp_replace

df_regex_replace = (
    df_new_col
    .withColumn("Item_Fat_Content", regexp_replace(col("Item_Fat_Content"), "Regular", "Reg"))
    .withColumn("Item_Fat_Content", regexp_replace(col("Item_Fat_Content"), "Low Fat", "Lfat"))
)

display(df_regex_replace)

In [0]:
from pyspark.sql.functions import when

df_when = df_new_col.withColumn(
    "Item_Fat_Content",
    when(col("Item_Fat_Content") == "Regular", "Reg")
    .when(col("Item_Fat_Content") == "Low Fat", "Lf")
    .otherwise("Unknown")
)

display(df_when)

### TYPE CASTING

In [0]:
display(df.limit(5))

In [0]:
df_casting = df.withColumn("Item_Weight", col("Item_Weight").cast("string"))
df_casting.printSchema()

### SORT / ORDERBY

In [0]:
# orderBy() dan sort() memiliki fungsi yang SAMA PERSIS.
# Default sorting adalah ascending.
# Untuk descending, gunakan col("kolom").desc() atau ascending=False.
# Dapat mengurutkan berdasarkan satu atau beberapa kolom.
# Dapat menggabungkan arah sorting yang berbeda (asc() dan desc()) untuk tiap kolom.
# Sorting di Spark adalah operasi yang mahal karena memerlukan shuffle, sehingga umumnya dilakukan setelah filter dan transformasi lain, atau menjelang output akhir.

In [0]:
display(df.limit(5))

In [0]:
df_sorted = df.orderBy(col("Item_Weight").desc())
display(df_sorted.limit(5))

In [0]:
df_sorted_mix = df.orderBy(col("Item_Weight").desc(), col("Item_Visibility").asc())
display(df_sorted_mix.limit(7))

### DROP
drop() di PySpark digunakan untuk menghapus sejumlah columns dari DataFrame.

In [0]:
# select() → tentukan kolom yang ingin dipertahankan.
# drop() → tentukan kolom yang ingin dibuang.
# Jadi select dan drop sebenarnya menshare trait yang mirip

# Kita akan menghapus column Item Weight dan Item Visibilty dan tetap menyimpan semua column lainnya
df_drop = df.drop(col("Item_Weight"), col("Item_Visibility"))

# Bisa juga tanpa col
# df_drop = df.drop("Item_Weight", "Item_Visibility")

display(df_drop.limit(7))